# 02 — Config and Sites (robust)

Loads `configs/run.yml` and writes a normalized `sites.json` table.

Supports:
- **Preferred v2**: `run:` + `sites: [ ... ]`
- **Legacy v1**: `study:` + `sites: {id: {...}}` + optional `controls:`

Outputs:
- `outputs/02_config_and_sites/run_config.json`
- `outputs/02_config_and_sites/sites.json`
- `outputs/02_config_and_sites/manifest.json`


In [ ]:
import json, hashlib
from pathlib import Path
from datetime import datetime, timezone
import yaml

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()

def sha256_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

def sha256_file(p: Path) -> str:
    return sha256_bytes(p.read_bytes())

def write_json(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2), encoding="utf-8")

def manifest_base(step: str, config_paths=None):
    m = {
        "step": step,
        "created_at_utc": utc_now(),
        "configs": [],
        "artifacts": []
    }
    for cp in (config_paths or []):
        if cp and cp.exists():
            m["configs"].append({
                "path": str(cp),
                "sha256": sha256_file(cp)
            })
    return m

def add_artifact(manifest: dict, path: Path, kind: str):
    if path.exists():
        manifest["artifacts"].append({
            "kind": kind,
            "path": str(path),
            "sha256": sha256_file(path)
        })


In [ ]:
# --- Resolve repo root robustly (works in GitHub Actions nbconvert and locally) ---
cwd = Path.cwd().resolve()

def find_repo_root(start: Path) -> Path:
    """Walk upwards until we find a directory that looks like the repo root."""
    cur = start
    for _ in range(8):
        if (cur / "configs").exists() and (cur / "notebooks").exists() and (cur / "scripts").exists():
            return cur
        if (cur / ".git").exists() or (cur / ".github").exists():
            # Often present at repo root; still require configs/ to be safe
            if (cur / "configs").exists():
                return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    return start

ROOT = find_repo_root(cwd)
CONFIGS = ROOT / "configs"
OUTPUTS = ROOT / "outputs"

print("CWD:", cwd)
print("ROOT:", ROOT)
print("CONFIGS:", CONFIGS)
print("OUTPUTS:", OUTPUTS)


In [ ]:
step = "02_config_and_sites"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

cfg_path = CONFIGS / "run.yml"
if not cfg_path.exists():
    # extra debug help
    print("ERROR: configs/run.yml not found")
    print("Listing ROOT:")
    for p in sorted(ROOT.glob("*") ):
        print(" -", p.name)
    if CONFIGS.exists():
        print("Listing CONFIGS:")
        for p in sorted(CONFIGS.glob("*") ):
            print(" -", p.name)
    raise FileNotFoundError("configs/run.yml not found")

cfg = yaml.safe_load(cfg_path.read_text(encoding="utf-8"))
write_json(out / "run_config.json", cfg)

# --- Normalize sites into a single flat table ---
rows = []

# Preferred schema (v2): sites is a LIST
if isinstance(cfg.get("sites"), list):
    for s in cfg.get("sites", []):
        sid = s.get("id") or s.get("site_id") or s.get("name")
        rows.append({
            "site_id": sid,
            "role": (s.get("role") or "context"),
            "name": s.get("name"),
            "lat": s.get("lat"),
            "lon": s.get("lon"),
        })

# Legacy schema (v1): sites/controls are DICTS
else:
    for k, v in (cfg.get("sites") or {}).items():
        role = v.get("role") or "context"
        rows.append({"site_id": k, "role": role, **v})
    for k, v in (cfg.get("controls") or {}).items():
        rows.append({"site_id": k, "role": "control", **v})

# basic validation
missing = [r for r in rows if r.get("lat") is None or r.get("lon") is None or not r.get("name")]
if missing:
    print("WARNING: Some sites are missing name/lat/lon:")
    for r in missing:
        print(" -", r)

write_json(out / "sites.json", rows)

manifest = manifest_base(step, config_paths=[cfg_path])
add_artifact(manifest, out / "run_config.json", "config_json")
add_artifact(manifest, out / "sites.json", "site_table")
write_json(out / "manifest.json", manifest)

# Print summary
if "study" in cfg:
    st = cfg.get("study", {})
    print("Study window:", st.get("date_start"), "to", st.get("date_end"), st.get("timezone"))
else:
    rn = cfg.get("run", {})
    print("Run window:", rn.get("date_from"), "to", rn.get("date_to"), rn.get("timezone"))

print("Sites normalised:", len(rows))
for r in rows:
    print("-", r.get("site_id"), r.get("name"), r.get("lat"), r.get("lon"), r.get("role"))

print("Wrote:", out / "manifest.json")
